### Utility Functions

In [ ]:
# This part helps with stability on multi-GPU systems with great inbalance between the GPUs (eg. integrated vs discrete gpu)
# IMPORTANT must be done before importing torch, else session must be restarted
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch

for i in range(torch.cuda.device_count()):
    free = torch.cuda.mem_get_info(i)[0]
    total = torch.cuda.mem_get_info(i)[1]

    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Free:  {free / 1024**3:.2f} GB")
    print(f"  Total: {total / 1024**3:.2f} GB")

In [ ]:
from util import clear_folder

# Clears results of the last run, not necessary, just to reduce folder size

clear_folder("./results")

In [ ]:
from util import clear_cuda_cache

# If model gets stuck during training, uncomment the following line

clear_cuda_cache()

# Training Pipeline

## Load Dataset

This portion loads dataset, and assigns id for each label

#### English

This loads english version of the dataset

In [ ]:
from util import load_datasets_from_hf

label2id={"ham": 0, "spam": 1}

train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-combined",
    split="train",
    label2id=label2id
)


#### Slovenian

This loads slovenian version of the dataset

In [ ]:
from util import load_datasets_from_hf

label2id={"ham": 0, "spam": 1}

train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-slovene",
    split="train",
    label2id=label2id
)


## Model settings 

### XLM-RoBERTA

In [ ]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "FacebookAI/xlm-roberta-base"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


### TinyBert

### Multilingual TinyBERT (recommended)

In [ ]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer

# Multilingual small model (Tiny-like) distilled from XLM-R; supports Slovene well
model_name = "nreimers/mMiniLMv2-L6-H384-distilled-from-XLMR-Large"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


In [ ]:

from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


In [ ]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "huawei-noah/TinyBERT_General_4L_312D"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


### mBERT

In [ ]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "bert-base-multilingual-cased"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


## Train

In [ ]:
from datetime import datetime
from util import clear_folder

clear_folder("./results")

trainer.train()

model_save_folder = "trained_models" 

model_save_name = model_name.split("/")[-1] + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
trainer.save_model(os.path.join(model_save_folder, model_save_name))
tokenizer.save_pretrained(os.path.join(model_save_folder, model_save_name))

In [ ]:
trainer.evaluate()

### Data Visualization

In [ ]:
import graphs
import importlib

importlib.reload(graphs)

graphs.generate_all_plots("./results", "./graphs" + "/" + model_name.split("/")[-1])

# Testing pipeline

In [ ]:
from util import load_latest_trained_model_and_tokenizer, build_trainer

model, tokenizer, latest_dir = load_latest_trained_model_and_tokenizer(
    trained_models_root="./trained_models",
    num_labels=2,
 )
print("Loaded:", latest_dir)


trainer = build_trainer(model, train_dataset, val_dataset)

## Load a previously trained model (skip training)

If you already trained a model once, you can reload it from `./trained_models` and run evaluation/zero-shot testing without training again.

# Zero-shot testing (EN -> SL)

Evaluate the model trained on the English dataset on the Slovenian dataset without any further training.

In [ ]:
from util import load_datasets_from_hf, tokenize_dataset


sl_train_ds, sl_val_ds, sl_test_ds, sl_id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-slovene",
    split="train",
    label2id=label2id,
    train_size=0.8,
    val_size=0.1,
    seed=42,
 )

sl_test_ds_tok = tokenize_dataset(sl_test_ds, tokenizer)

trainer.eval_dataset = sl_test_ds_tok

zero_shot_metrics = trainer.evaluate()


In [ ]:
from util import load_datasets_from_hf, tokenize_dataset


sl_train_ds, sl_val_ds, sl_test_ds, sl_id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-combined",
    split="train",
    label2id=label2id,
    train_size=0.8,
    val_size=0.1,
    seed=42,
 )

sl_test_ds_tok = tokenize_dataset(sl_test_ds, tokenizer)

trainer.eval_dataset = sl_test_ds_tok

zero_shot_metrics = trainer.evaluate()


In [ ]:
from util import predict_labels
import graphs

labels_in_order = [sl_id2label[i] for i in range(len(sl_id2label))]

preds, labels = predict_labels(trainer, sl_test_ds_tok)
graph_dir = os.path.join("./graphs", os.path.basename(latest_dir))
graphs.plot_confusion_matrix(
    y_true=labels,
    y_pred=preds,
    labels=labels_in_order,
    save_path=os.path.join(graph_dir, "zero_shot_en_to_sl_confusion_matrix.png"),
)
print(f"Saved confusion matrix to {os.path.join(graph_dir, 'zero_shot_en_to_sl_confusion_matrix.png')}")

## Few-shot adaptation (mBERT)

Fine-tune the mBERT model (trained on English) on a small Slovenian sample to adapt it.

**Validation holdout suggestion:** when using very small few-shot sizes, keep a small validation holdout (e.g. 10-20% of the few-shot examples or 50–100 examples) to monitor overfitting. The code below will automatically use the existing Slovenian validation split if available, otherwise it will create a holdout from the sampled few-shot set.

In [ ]:
# Few-shot evaluation and comparison using the existing build_trainer helper
import importlib
import util
importlib.reload(util)
from util import load_datasets_from_hf, tokenize_dataset, load_model_and_tokenizer, load_trained_model_and_tokenizer, build_trainer, predict_labels
import os
from datetime import datetime
import csv
import graphs

few_shot_ks = [0, 10, 50, 100, 200]
mb_model_name = "bert-base-multilingual-cased"
trained_root = "./trained_models"
results_dir = "./results"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(trained_root, exist_ok=True)
label2id = {"ham": 0, "spam": 1}

# Ensure English and Slovenian splits are available
try:
    train_dataset
    val_dataset
    test_dataset
except NameError:
    label2id = {"ham": 0, "spam": 1}
    train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf("daviiiidcpp1/sms-spam-combined", split="train", label2id=label2id)

try:
    sl_train_ds
    sl_val_ds
    sl_test_ds
    sl_id2label
except NameError:
    sl_train_ds, sl_val_ds, sl_test_ds, sl_id2label = load_datasets_from_hf("daviiiidcpp1/sms-spam-slovene", split="train", label2id=label2id, train_size=0.8, val_size=0.1, seed=42)

# Prefer the explicitly named saved mBERT checkpoint; otherwise train a fresh English base.
base_model_name = "bert-base-multilingual-cased2026-05-07_22-06-34"
base_dir = os.path.join(trained_root, base_model_name)

if os.path.isdir(base_dir):
    model, tokenizer = load_trained_model_and_tokenizer(base_dir, num_labels=2)
    print("Using saved mBERT base checkpoint:", base_dir)
else:
    model, tokenizer = load_model_and_tokenizer(mb_model_name, num_labels=2)
    train_tok = tokenize_dataset(train_dataset, tokenizer)
    val_tok = tokenize_dataset(val_dataset, tokenizer)
    base_trainer = build_trainer(
        model,
        train_tok,
        val_tok,
        learning_rate=2e-5,
        num_epochs=1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        output_dir=os.path.join(results_dir, mb_model_name.split("/")[-1] + "_base"),
        logging_steps=50,
        save_total_limit=1,
    )
    base_trainer.train()
    base_dir = os.path.join(trained_root, mb_model_name.split("/")[-1] + "_base_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))
    base_trainer.save_model(base_dir)
    tokenizer.save_pretrained(base_dir)
    print("Trained and saved English base mBERT to:", base_dir)

# Tokenize Slovenian test once for all runs
sl_test_ds_tok_full = tokenize_dataset(sl_test_ds, tokenizer)
results_csv = os.path.join(results_dir, mb_model_name.split("/")[-1] + "_fewshot_results.csv")
labels_in_order = [sl_id2label[i] for i in range(len(sl_id2label))]

with open(results_csv, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["few_shot_k", "accuracy", "precision", "recall", "f1", "train_size", "val_size", "checkpoint_dir"])

for k in few_shot_ks:
    print(f"--- few_shot_k={k} ---")
    if k == 0:
        base_model, base_tokenizer = load_trained_model_and_tokenizer(base_dir, num_labels=2)
        base_eval_trainer = build_trainer(
            base_model,
            train_tok if 'train_tok' in globals() else tokenize_dataset(train_dataset, base_tokenizer),
            val_tok if 'val_tok' in globals() else tokenize_dataset(val_dataset, base_tokenizer),
            output_dir=os.path.join(results_dir, mb_model_name.split("/")[-1] + "_zero_shot_eval"),
        )
        base_eval_trainer.eval_dataset = sl_test_ds_tok_full
        metrics = base_eval_trainer.evaluate()
        preds, labels = predict_labels(base_eval_trainer, sl_test_ds_tok_full)
        save_name = mb_model_name.split("/")[-1] + "_zero_shot_eval_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        graph_dir = os.path.join("./graphs", save_name)
        os.makedirs(graph_dir, exist_ok=True)
        graphs.plot_confusion_matrix(y_true=labels, y_pred=preds, labels=labels_in_order, save_path=os.path.join(graph_dir, "confusion_matrix.png"))
        with open(results_csv, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow([k, metrics["eval_accuracy"], metrics["eval_precision"], metrics["eval_recall"], metrics["eval_f1"], 0, len(sl_val_ds), base_dir])
        continue

    k_use = min(k, len(sl_train_ds))
    few_shot_train = sl_train_ds.shuffle(seed=42).select(range(k_use))

    if len(sl_val_ds) >= 20:
        few_shot_val = sl_val_ds.shuffle(seed=43).select(range(min(100, len(sl_val_ds))))
    else:
        holdout = max(1, int(0.1 * k_use))
        holdout = min(holdout, max(0, k_use - 1))
        if holdout > 0:
            few_shot_val = few_shot_train.select(range(holdout))
            few_shot_train = few_shot_train.select(range(holdout, k_use))
        else:
            few_shot_val = few_shot_train

    few_shot_train_tok = tokenize_dataset(few_shot_train, tokenizer)
    few_shot_val_tok = tokenize_dataset(few_shot_val, tokenizer)

    model_k, tokenizer_k = load_trained_model_and_tokenizer(base_dir, num_labels=2)
    trainer_k = build_trainer(
        model_k,
        few_shot_train_tok,
        few_shot_val_tok,
        learning_rate=2e-5,
        num_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        output_dir=os.path.join(results_dir, f"{mb_model_name.split('/')[-1]}_fewshot_k{k}"),
        logging_steps=50,
        save_total_limit=2,
    )
    trainer_k.train()
    trainer_k.eval_dataset = sl_test_ds_tok_full
    metrics_k = trainer_k.evaluate()
    preds_k, labels_k = predict_labels(trainer_k, sl_test_ds_tok_full)

    save_name_k = mb_model_name.split("/")[-1] + f"_fewshot_k{k}_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    save_dir_k = os.path.join(trained_root, save_name_k)
    trainer_k.save_model(save_dir_k)
    tokenizer.save_pretrained(save_dir_k)

    graph_dir_k = os.path.join("./graphs", save_name_k)
    os.makedirs(graph_dir_k, exist_ok=True)
    graphs.plot_confusion_matrix(y_true=labels_k, y_pred=preds_k, labels=labels_in_order, save_path=os.path.join(graph_dir_k, "confusion_matrix.png"))

    with open(results_csv, "a", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([k, metrics_k["eval_accuracy"], metrics_k["eval_precision"], metrics_k["eval_recall"], metrics_k["eval_f1"], len(few_shot_train_tok), len(few_shot_val_tok), save_dir_k])

    print("Saved checkpoint to:", save_dir_k)
    print("Saved confusion matrix to:", os.path.join(graph_dir_k, "confusion_matrix.png"))

print("All few-shot runs completed. Results saved to:", results_csv)

In [ ]:
import importlib
import graphs

importlib.reload(graphs)

metrics = ["accuracy", "precision", "recall", "f1"]
few_shot_results_csv = os.path.join("./results", "bert-base-multilingual-cased_fewshot_results.csv")

for metric in metrics:
    plot_path = graphs.plot_fewshot_comparison(few_shot_results_csv, save_path="./graphs/bert-base-multilingual-cased-fewshot_k_comparison", metric=metric)
    print(f"Saved few-shot comparison plot for {metric} to:", plot_path)

### Few-shot adaptation (TinyBert)

Fine-tune the TinyBERT model on a small Slovenian sample to adapt it.

**Validation holdout suggestion:** when using very small few-shot sizes, keep a small validation holdout (e.g. 10-20% of the few-shot examples or 50–100 examples) to monitor overfitting. The code below will automatically use the existing Slovenian validation split if available, otherwise it will create a holdout from the sampled few-shot set.

In [ ]:
# Few-shot evaluation and comparison using the existing build_trainer helper
import importlib
import util
importlib.reload(util)
from util import load_datasets_from_hf, tokenize_dataset, load_model_and_tokenizer, load_trained_model_and_tokenizer, build_trainer, predict_labels
import os
from datetime import datetime
import csv
import graphs

few_shot_ks = [0, 10, 50, 100, 200]
tb_model_name = "TinyBERT_General_4L_312D"
trained_root = "./trained_models"
results_dir = "./results"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(trained_root, exist_ok=True)
label2id = {"ham": 0, "spam": 1}

# Ensure English and Slovenian splits are available
try:
    train_dataset
    val_dataset
    test_dataset
except NameError:
    label2id = {"ham": 0, "spam": 1}
    train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf("daviiiidcpp1/sms-spam-combined", split="train", label2id=label2id)

try:
    sl_train_ds
    sl_val_ds
    sl_test_ds
    sl_id2label
except NameError:
    sl_train_ds, sl_val_ds, sl_test_ds, sl_id2label = load_datasets_from_hf("daviiiidcpp1/sms-spam-slovene", split="train", label2id=label2id, train_size=0.8, val_size=0.1, seed=42)

# Prefer the explicitly named saved TinyBERT base checkpoint; otherwise train a fresh base.
base_model_name = "TinyBERT_General_4L_312D_base_2026-05-09_19-58-24"
base_dir = os.path.join(trained_root, base_model_name)

if os.path.isdir(base_dir):
    model, tokenizer = load_trained_model_and_tokenizer(base_dir, num_labels=2)
    print("Using saved TinyBERT base checkpoint:", base_dir)
else:
    model, tokenizer = load_model_and_tokenizer("huawei-noah/TinyBERT_General_4L_312D", num_labels=2)
    train_tok = tokenize_dataset(train_dataset, tokenizer)
    val_tok = tokenize_dataset(val_dataset, tokenizer)
    base_trainer = build_trainer(
        model,
        train_tok,
        val_tok,
        learning_rate=2e-5,
        num_epochs=1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        output_dir=os.path.join(results_dir, tb_model_name + "_base"),
        logging_steps=50,
        save_total_limit=1,
    )
    base_trainer.train()
    base_dir = os.path.join(trained_root, tb_model_name + "_base_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))
    base_trainer.save_model(base_dir)
    tokenizer.save_pretrained(base_dir)
    print("Trained and saved English base TinyBERT to:", base_dir)

# Tokenize Slovenian test once for all runs
sl_test_ds_tok_full = tokenize_dataset(sl_test_ds, tokenizer)
results_csv = os.path.join(results_dir, tb_model_name + "_fewshot_results.csv")
labels_in_order = [sl_id2label[i] for i in range(len(sl_id2label))]

with open(results_csv, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["few_shot_k", "accuracy", "precision", "recall", "f1", "train_size", "val_size", "checkpoint_dir"])

for k in few_shot_ks:
    print(f"--- few_shot_k={k} ---")
    if k == 0:
        base_model, base_tokenizer = load_trained_model_and_tokenizer(base_dir, num_labels=2)
        base_eval_trainer = build_trainer(
            base_model,
            train_tok if 'train_tok' in globals() else tokenize_dataset(train_dataset, base_tokenizer),
            val_tok if 'val_tok' in globals() else tokenize_dataset(val_dataset, base_tokenizer),
            output_dir=os.path.join(results_dir, tb_model_name + "_zero_shot_eval"),
        )
        base_eval_trainer.eval_dataset = sl_test_ds_tok_full
        metrics = base_eval_trainer.evaluate()
        preds, labels = predict_labels(base_eval_trainer, sl_test_ds_tok_full)
        save_name = tb_model_name + "_zero_shot_eval_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        graph_dir = os.path.join("./graphs", save_name)
        os.makedirs(graph_dir, exist_ok=True)
        graphs.plot_confusion_matrix(y_true=labels, y_pred=preds, labels=labels_in_order, save_path=os.path.join(graph_dir, "confusion_matrix.png"))
        with open(results_csv, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow([k, metrics["eval_accuracy"], metrics["eval_precision"], metrics["eval_recall"], metrics["eval_f1"], 0, len(sl_val_ds), base_dir])
        continue

    k_use = min(k, len(sl_train_ds))
    few_shot_train = sl_train_ds.shuffle(seed=42).select(range(k_use))

    if len(sl_val_ds) >= 20:
        few_shot_val = sl_val_ds.shuffle(seed=43).select(range(min(100, len(sl_val_ds))))
    else:
        holdout = max(1, int(0.1 * k_use))
        holdout = min(holdout, max(0, k_use - 1))
        if holdout > 0:
            few_shot_val = few_shot_train.select(range(holdout))
            few_shot_train = few_shot_train.select(range(holdout, k_use))
        else:
            few_shot_val = few_shot_train

    few_shot_train_tok = tokenize_dataset(few_shot_train, tokenizer)
    few_shot_val_tok = tokenize_dataset(few_shot_val, tokenizer)

    model_k, tokenizer_k = load_trained_model_and_tokenizer(base_dir, num_labels=2)
    trainer_k = build_trainer(
        model_k,
        few_shot_train_tok,
        few_shot_val_tok,
        learning_rate=2e-5,
        num_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        output_dir=os.path.join(results_dir, f"{tb_model_name}_fewshot_k{k}"),
        logging_steps=50,
        save_total_limit=2,
    )
    trainer_k.train()
    trainer_k.eval_dataset = sl_test_ds_tok_full
    metrics_k = trainer_k.evaluate()
    preds_k, labels_k = predict_labels(trainer_k, sl_test_ds_tok_full)

    save_name_k = tb_model_name + f"_fewshot_k{k}_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    save_dir_k = os.path.join(trained_root, save_name_k)
    trainer_k.save_model(save_dir_k)
    tokenizer.save_pretrained(save_dir_k)

    graph_dir_k = os.path.join("./graphs", save_name_k)
    os.makedirs(graph_dir_k, exist_ok=True)
    graphs.plot_confusion_matrix(y_true=labels_k, y_pred=preds_k, labels=labels_in_order, save_path=os.path.join(graph_dir_k, "confusion_matrix.png"))

    with open(results_csv, "a", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([k, metrics_k["eval_accuracy"], metrics_k["eval_precision"], metrics_k["eval_recall"], metrics_k["eval_f1"], len(few_shot_train_tok), len(few_shot_val_tok), save_dir_k])

    print("Saved checkpoint to:", save_dir_k)
    print("Saved confusion matrix to:", os.path.join(graph_dir_k, "confusion_matrix.png"))

print("All few-shot runs completed. Results saved to:", results_csv)

In [ ]:
import importlib
import graphs

importlib.reload(graphs)

metrics = ["accuracy", "precision", "recall", "f1"]
few_shot_results_csv = os.path.join("./results", "TinyBERT_General_4L_312D_fewshot_results.csv")

for metric in metrics:
    plot_path = graphs.plot_fewshot_comparison(few_shot_results_csv, save_path="./graphs/TinyBERT_fewshot_k_comparison", metric=metric)
    print(f"Saved few-shot comparison plot for {metric} to:", plot_path)

# Model Comparison: BERT vs TinyBERT

Compare few-shot performance across k values for mBERT and TinyBERT models side-by-side.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Create output directory for comparison graphs
comparison_dir = "./graphs/bert_tinybert_k_comparison"
os.makedirs(comparison_dir, exist_ok=True)

# Load BERT and TinyBERT results
mbert_csv = "./results/bert-base-multilingual-cased_fewshot_results.csv"
tinybert_csv = "./results/TinyBERT_General_4L_312D_fewshot_results.csv"

# Read CSV files
mbert_df = pd.read_csv(mbert_csv)
tinybert_df = pd.read_csv(tinybert_csv)

# Sort by few_shot_k for proper x-axis ordering
mbert_df = mbert_df.sort_values("few_shot_k").reset_index(drop=True)
tinybert_df = tinybert_df.sort_values("few_shot_k").reset_index(drop=True)

# Create comparison plots for each metric
metrics = ["accuracy", "precision", "recall", "f1"]
for metric in metrics:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot both models
    ax.plot(mbert_df["few_shot_k"], mbert_df[metric], marker='o', linewidth=2, label="mBERT", color='#1f77b4')
    ax.plot(tinybert_df["few_shot_k"], tinybert_df[metric], marker='s', linewidth=2, label="TinyBERT", color='#ff7f0e')
    
    ax.set_xlabel("Few-shot k", fontsize=12)
    ax.set_ylabel(metric.capitalize(), fontsize=12)
    ax.set_title(f"BERT vs TinyBERT: {metric.capitalize()} by k", fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(mbert_df["few_shot_k"].unique())
    
    # Save figure
    save_path = os.path.join(comparison_dir, f"bert_tinybert_{metric}.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved comparison plot for {metric} to: {save_path}")

# Create a combined comparison table showing differences
comparison_data = []
for idx, row in mbert_df.iterrows():
    k = row["few_shot_k"]
    mbert_row = mbert_df[mbert_df["few_shot_k"] == k].iloc[0]
    tinybert_row = tinybert_df[tinybert_df["few_shot_k"] == k].iloc[0]
    
    comparison_data.append({
        "k": k,
        "mBERT_accuracy": round(mbert_row["accuracy"], 4),
        "TinyBERT_accuracy": round(tinybert_row["accuracy"], 4),
        "accuracy_diff": round(tinybert_row["accuracy"] - mbert_row["accuracy"], 4),
        "mBERT_f1": round(mbert_row["f1"], 4),
        "TinyBERT_f1": round(tinybert_row["f1"], 4),
        "f1_diff": round(tinybert_row["f1"] - mbert_row["f1"], 4),
    })

comparison_table = pd.DataFrame(comparison_data)
comparison_csv_path = os.path.join(comparison_dir, "bert_tinybert_comparison.csv")
comparison_table.to_csv(comparison_csv_path, index=False)
print(f"\nComparison table saved to: {comparison_csv_path}")
print("\nComparison Summary:")
print(comparison_table.to_string(index=False))